# 06 — Systematics and Robustness

Quantify sensitivity to assumptions: normalization, scan range, transfer factors, and asymmetric uncertainty choices.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
import itertools
import matplotlib.pyplot as plt

In [ ]:
def run_robustness_grid(scan_fn, op, sigma_sm_values, ranges, n_values):
    rows=[]
    for sig_sm in sigma_sm_values:
        for (lo,hi) in ranges:
            for n in n_values:
                df=scan_fn(op=op, sigma_sm_hww=sig_sm, wc_min=lo, wc_max=hi, n=n)
                bf=df.loc[df['chi2_hww'].idxmin()]
                rows.append({'sigma_sm_hww':sig_sm,'wc_min':lo,'wc_max':hi,'n':n,'best_wc':bf['wc'],'chi2_min':bf['chi2_hww']})
    return pd.DataFrame(rows)

In [ ]:
ROBUSTNESS_CHECKLIST=[
 'vary sigma_sm_hww by theory uncertainty',
 'vary wc scan range',
 'vary bin-proxy/transfer factor for m_jj>1TeV mapping',
 'compare asymmetric vs symmetric likelihood treatment',
 'check dependence on operator subset in 2D scans'
]
for i,x in enumerate(ROBUSTNESS_CHECKLIST,1):
    print(i,x)